In [1]:
import os
import pandas as pd
import numpy as np
import re
import random
import itertools
import jellyfish
from collections import defaultdict

csv = pd.read_csv("Data/output.csv", dtype = {"ZipCode": str})
csv.insert(0, "id",[i for i in range(len(csv))])

csv.head()

,id,StreetName,Address,ZipCode,X_COORD,Y_COORD
0,0,BLACK ROCK RD,35,06903,-73.557934,41.178028
1,1,BLACK ROCK RD,79,06903,-73.559713,41.177148
2,2,BLACK ROCK RD,86,06903,-73.559469,41.176574
3,3,BLACK ROCK RD,99,06903,-73.560948,41.176826
4,4,BLACK ROCK RD,111,06903,-73.560947,41.176244


*Exploration of directional prefixes/suffixes and the hard-negative score breakdown moved to **Diagnostics** at the bottom of this notebook.*

In [5]:
new_df = csv.copy()

def get_directional(text):
    valid_dirs = ["N","S","E","W"]
    directional = None
    split_txt = text.split()
    directional =  split_txt[0] if split_txt[0] in valid_dirs else directional
    directional = split_txt[-1] if split_txt[-1] in valid_dirs else directional
    return directional
new_df["directional"] = new_df["StreetName"].apply(get_directional)

def drop_directional(text):
    valid_dirs = ["N","S","E","W"]
    split_txt = text.split()
    split_txt = split_txt[1:] if split_txt[0] in valid_dirs else split_txt
    split_txt = split_txt[:-1] if split_txt[-1] in valid_dirs else split_txt
    return " ".join(split_txt)
new_df["StreetName"] = new_df["StreetName"].apply(drop_directional)


def get_ext(text):
    valid_ext = ["EXT"]
    ext = False
    split_txt = text.split()
    ext =  True if split_txt[0] in valid_ext else ext
    ext = True if split_txt[-1] in valid_ext else ext
    return ext
new_df["is_extensional"] = new_df["StreetName"].apply(get_ext)

def drop_ext(text):
    valid_ext = ["EXT"]
    split_txt = text.split()
    split_txt = split_txt[1:] if split_txt[0] in valid_ext else split_txt
    split_txt = split_txt[:-1] if split_txt[-1] in valid_ext else split_txt
    return " ".join(split_txt)
new_df["StreetName"] = new_df["StreetName"].apply(drop_ext)

known_suffixes = ["RD", "ST", "AVE", "DR", "LN", "PL", "CT", "CIR", "TER", "TRL", "BLVD", "WAY", "QUAY", "PARK", "TPKE", "HOLW", "GRV", "DOCK", "PATH", "RUN", "LNDG", "PLZ", "WALK", "PT"]
new_df["street_extension"] = new_df["StreetName"].apply(
    lambda x: x.split()[-1] if x.split()[-1] in known_suffixes else None
)
new_df["StreetName"] = new_df["StreetName"].apply(
    lambda x: " ".join(x.split()[:-1] if x.split()[-1] in known_suffixes else x.split())
) 


new_df
# new_df[~new_df["Directional"].isna()]

,id,StreetName,Address,ZipCode,X_COORD,Y_COORD,directional,is_extensional,street_extension
0,0,BLACK ROCK,35,06903,-73.557934,41.178028,None,False,RD
1,1,BLACK ROCK,79,06903,-73.559713,41.177148,None,False,RD
2,2,BLACK ROCK,86,06903,-73.559469,41.176574,None,False,RD
3,3,BLACK ROCK,99,06903,-73.560948,41.176826,None,False,RD
4,4,BLACK ROCK,111,06903,-73.560947,41.176244,None,False,RD
...,...,...,...,...,...,...,...,...,...
28289,28289,STILLWATER,1019,06902,-73.557396,41.077495,None,False,RD
28290,28290,STILLWATER,1025,06902,-73.557560,41.077722,None,False,RD
28291,28291,STILLWATER,1031,06902,-73.557236,41.077921,None,False,RD
28292,28292,STILLWATER,1041,06902,-73.557844,41.078156,None,False,RD


In [6]:
from pprint import pprint

suffix_abbrev = pd.read_csv("Data/usps_pub28_street_suffix_abbreviations.csv")

sfx_abrev_lookup = suffix_abbrev.groupby("standard_abbreviation")["commonly_used"].apply(list).to_dict()
# pprint(sfx_abrev_lookup["PL"])

for key, variants in sfx_abrev_lookup.items():
    index = variants.index(key) if key in variants else None
    if index is not  None:
        sfx_abrev_lookup[key] = sfx_abrev_lookup[key][:index] + sfx_abrev_lookup[key][index+1:]
sfx_abrev_lookup = {
    key:variants for key, variants in sfx_abrev_lookup.items() if len(variants) > 0
}

directional_lookup = {
    "N": ["NORTH"],
    "S": ["SOUTH"],
    "E": ["EAST"],
    "W": ["WEST"],
}

In [7]:

def _generate_mask(street_name, pct_flip = 0.1):
    # assert sum(pct_op) == 1 and len(pct_op) == 2
    n = len(street_name)
    mask = [1 if random.random() <= pct_flip else 0 for _ in range(n)]
    while sum(mask) > n//2:
       mask = [1 if random.random() <= pct_flip else 0 for _ in range(n)] 

    #1 = Removal
    #2 = Replacement
    for i, res in enumerate(mask):
        if res == 1:
            if random.random() > 0.5: mask[i] = 2
    return mask

def update_string(street_name, pct2):
    new_street_name = ""
    for i,op in enumerate(_generate_mask(street_name, pct2)):
        c = street_name[i]
        if op == 0:
            new_street_name += street_name[i]
        elif op == 1:
            pass
        elif op == 2:
            if ord('A') <= ord(c) <= ord('Z'):
                new_street_name += random.choice([chr(c) for c in range(ord('A'), ord('Z') + 1)])
            if ord('a') <= ord(c) <= ord('z'):
                new_street_name += random.choice([chr(c) for c in range(ord('a'), ord('z') + 1)])
    return new_street_name


In [8]:
def _corrupt_row(row, pct1=0.7, pct2=0.1, pct3=0.7):

    new_row = row.copy()
    for row_name, lookup in zip(["street_extension", "directional"], [sfx_abrev_lookup, directional_lookup]):
        if row[row_name] in lookup and random.random() < pct1:
            choices = lookup[row[row_name]]
            new_row[row_name] = random.choice(choices)
    new_row["StreetName"] = update_string(row["StreetName"], pct2)
    return new_row

def corrupt_rows(df):
    corrupted_rows = [_corrupt_row(row) for _, row in df.iterrows()]
    new_df = pd.DataFrame(corrupted_rows)
    new_df = new_df.rename(columns = {"id": "original_id"})
    return new_df

corrupted_df = corrupt_rows(new_df)

corrupted_df.head()

,original_id,StreetName,Address,ZipCode,X_COORD,Y_COORD,directional,is_extensional,street_extension
0,0,BACK ROCK,35,06903,-73.557934,41.178028,None,False,ROAD
1,1,HLACK ROCK,79,06903,-73.559713,41.177148,None,False,ROAD
2,2,BLACKGOCK,86,06903,-73.559469,41.176574,None,False,ROAD
3,3,BLACK ROCK,99,06903,-73.560948,41.176826,None,False,RD
4,4,AC RQCK,111,06903,-73.560947,41.176244,None,False,RD


In [9]:
#Compiling negative cases

NUM_NEGATIVE = 4

jw_lookup = {}
debug_jw = []

for address, group_df in new_df.groupby("Address"):
    for row1,row2 in itertools.combinations(group_df.to_dict('records'), 2):
        a,b, aid, bid = row1["StreetName"], row2["StreetName"], row1["id"], row2["id"]    
        score = 0
        debug_jw_tmp = [aid, bid]
        def _add_score(amt, condition = True, default = 0):
             global score
             score += amt if condition else default
             debug_jw_tmp.append(amt if condition else default)
        _add_score(jellyfish.jaro_winkler_similarity(a, b))
        _add_score(0.05) #Due to same Address
        _add_score(0.05, row1["directional"] != None and row1["directional"] == row2["directional"]) 
        _add_score(0.05, row1["street_extension"] != None and row1["street_extension"] == row2["street_extension"])
        _add_score(0.05, row1["is_extensional"] == True and row1["is_extensional"] == row2["is_extensional"]) 
        jw_lookup[(min(aid, bid), max(aid, bid))] = score

        debug_jw_tmp.append(sum(debug_jw_tmp[2:]))
        debug_jw.append(debug_jw_tmp)        

def all_scores_above(thresh = 0.7):
    all_scores = [[res, a, b] for (a,b), res in jw_lookup.items()]
    return [[a,b] for res,a,b in all_scores if res >= thresh]

train_df_lst = []
for (StreetName, street_extension, directional), lst in new_df.groupby(["StreetName", "street_extension", "directional"], dropna = False)["id"].apply(list).to_dict().items():
    if len(lst) < 2: continue
    train_df_lst.extend(
        [[min(a, b),max(a,b)] for a,b in itertools.combinations(random.sample(lst, min(len(lst), NUM_NEGATIVE)), 2)]
    )
train_df_lst.extend(all_scores_above(0.85))
print("length w/ dupes", len(train_df_lst))
train_df_lst = list(set(map(tuple, train_df_lst)))
print("length w/o dupes", len(train_df_lst))

length w/ dupes 14819
length w/o dupes 14818


In [11]:
#Time to build the freaking data pipeline
# print(len(train_df_lst)) #14818
# print(len(new_df)) #28294
# print(len(corrupted_df)) #28294

n = len(corrupted_df)

new_df
corrupted_df 
train_df_lst
def _generate_positive_dataset(amount = 10000):
    indexes = random.sample([i for i in range(len(new_df))], k=amount)
    good = new_df.iloc[indexes]
    bad = corrupted_df.iloc[indexes]
    combined = pd.merge(good, bad,
        left_index = True,
        right_index = True,
        suffixes = ("_a", "_b")
    )
    combined.rename(columns = {"id": "id_a", "original_id":"id_b"}, inplace = True)
    combined.drop(columns = ["X_COORD_a","Y_COORD_a", "X_COORD_b", "Y_COORD_b"], inplace = True)
    combined["label"] = 1

    return combined

def _generate_easy_negative_dataset(amount = 10000):
    good = new_df.iloc[
        random.choices([i for i in range(len(new_df))], k = amount)
    ].reset_index(drop = True)
    bad = corrupted_df.iloc[
        random.choices([i for i in range(len(corrupted_df))], k = amount)
    ].reset_index(drop = True)
    dataset = pd.merge(good, bad, left_index = True, right_index = True,  suffixes=("_a", "_b"))
    dataset.drop(columns = ["X_COORD_a","Y_COORD_a","X_COORD_b","Y_COORD_b"], inplace = True)
    dataset.rename(columns = {"id":"id_a", "original_id":"id_b"}, inplace = True)
    dataset["label"] = 0
    return dataset

def _generate_hard_negative_dataset(amount = 10000):
    chosen_neg_data = [random.sample(_, k = len(_)) for _ in random.sample(train_df_lst, k = amount)]
    clean_ids = [x[0] for x in chosen_neg_data]
    dirty_ids = [x[1] for x in chosen_neg_data]
    good = new_df.set_index("id").loc[clean_ids].reset_index()
    bad = corrupted_df.set_index("original_id").loc[dirty_ids].reset_index()
    dataset = pd.merge(good, bad, left_index = True, right_index = True, suffixes = ("_a", "_b"))
    dataset.drop(columns = ["X_COORD_a","Y_COORD_a","X_COORD_b","Y_COORD_b"], inplace = True)
    dataset.rename(columns = {"id":"id_a", "original_id":"id_b"}, inplace = True)
    dataset["label"] = 0

    return dataset

def _generate_training_data():
    positive_dataset = _generate_positive_dataset(len(train_df_lst))
    easy_neg_dataset = _generate_easy_negative_dataset(len(train_df_lst))
    hard_neg_dataset = _generate_hard_negative_dataset(len(train_df_lst))
    dataset = pd.concat([positive_dataset, easy_neg_dataset, hard_neg_dataset])
    dataset = dataset.sample(frac = 1).reset_index(drop = True)

    return dataset

dataset = _generate_training_data()
dataset

# positive_dataset = _generate_positive_dataset()
# easy_neg_dataset = _generate_easy_negative_dataset()
# hard_neg_dataset = _generate_hard_negative_dataset()
# print(len(new_df), len(corrupted_df), len(train_df_lst))
# pd.concat([positive_dataset.dtypes, easy_neg_dataset.dtypes, hard_neg_dataset.dtypes], axis = 1)

# print(set(positive_dataset.columns) == set(easy_neg_dataset.columns))
# print(set(easy_neg_dataset) == set(hard_neg_dataset))

,id_a,StreetName_a,Address_a,ZipCode_a,directional_a,is_extensional_a,street_extension_a,id_b,StreetName_b,Address_b,ZipCode_b,directional_b,is_extensional_b,street_extension_b,label
0,20448,GARDEN,24,06902,None,False,ST,870,STOE FNCE,34,06903,None,False,LN,0
1,15216,TREMONT,58,06906,None,False,AVE,15216,TREMONT,58,06906,None,False,AVN,1
2,13296,UPLAND,16,06906,None,False,RD,12977,POND,16,06902,None,False,RD,0
3,27242,SYLVAN KNOLL,112,06902,None,False,RD,5863,HIDDEN BROK,75,06907,None,False,DRIV,0
4,27469,HIGH RIDGE,885,06905,None,False,RD,6965,LNGRIDGE,885,06902,None,False,ROAD,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44449,3053,CHESTNUT HILL,217,06903,None,False,RD,12010,TERRACE,15,06905,None,False,AVEN,0
44450,27463,MANHATTAN,55,06902,None,False,ST,27463,MDNHDTTN,55,06902,None,False,ST,1
44451,13622,EAST,66,06905,None,False,LN,1073,WEST,66,06903,None,False,TRAIL,0
44452,20417,PERRY,14,06902,None,False,ST,7948,MULBRRY,14,06907,None,False,STRT,0


In [12]:
from sklearn.model_selection import train_test_split
train_ids, test_ids = train_test_split(range(len(new_df)), test_size=0.15, random_state=42)
train_ids, val_ids = train_test_split(train_ids, test_size=0.2, random_state=42)

train_dataset = dataset[dataset["id_a"].isin(train_ids) & dataset["id_b"].isin(train_ids)]
val_dataset = dataset[dataset["id_a"].isin(val_ids) & dataset["id_b"].isin(val_ids)] 
test_dataset = dataset[dataset["id_a"].isin(test_ids) & dataset["id_b"].isin(test_ids)]  


for i, df in enumerate([train_dataset, val_dataset, test_dataset]):
    n0 = (df['label'] == 0).sum()
    n1 = (df['label'] == 1).sum()

    # print(df[df['label'] == 1])
    df_0 = df.loc[df['label'] == 0]
    df_1 = df.loc[df['label'] == 1]
    

    # 9:1 0:1
    target_n1 = int(n0/9)

    if target_n1 <= n1:
        df_1 = df_1.sample(n = target_n1, random_state = 42)

    if i == 0:
        train_dataset = pd.concat([df_0, df_1]).sample(frac = 1, random_state = 42).reset_index(drop = True)
    elif i == 1:
        val_dataset = pd.concat([df_0, df_1]).sample(frac = 1, random_state = 42).reset_index(drop = True)
    elif i == 2:
        test_dataset = pd.concat([df_0, df_1]).sample(frac = 1, random_state = 42).reset_index(drop = True)

print(len(train_dataset), len(val_dataset), len(test_dataset))

train_dataset.to_parquet("Data/train.parquet", index = False)
val_dataset.to_parquet("Data/val.parquet", index = False)
test_dataset.to_parquet("Data/test.parquet", index = False)

15225 977 766


## Diagnostics

One-off checks kept out of the pipeline above. Paths are relative to this notebook's folder.

### Directional positions

**Problem:** We needed to know where directionals sit in Stamford street names before parsing them out.  
**Check:** Frequency of `N/S/E/W…` as last word and as first word.  
**Result:** Suffix: W 248, E 199, S 172, N 87 · Prefix: W 358, E 260, N 46, S 32

```python
suffix_df = csv.copy()
suffix_df["suffix"] = suffix_df["StreetName"].str.split().str[-1]
suffix_frequencies = suffix_df["suffix"].value_counts()
print(suffix_frequencies.keys())
suffix_df["suffix_freq"] = suffix_df["suffix"].map(suffix_frequencies)
suffix_df = suffix_df.drop_duplicates(subset = ["suffix"])
suffix_df = suffix_df[suffix_df["suffix"].isin(["N","S","E","W","NE","NW","SE","SW"])]
suffix_df = suffix_df.sort_values(by = "suffix_freq", ascending= False)
suffix_df

prefix_df = csv.copy()
prefix_df["prefix"] = prefix_df["StreetName"].str.split().str[0]
prefix_frequencies = prefix_df["prefix"].value_counts()
print(prefix_frequencies.keys())
prefix_df["prefix_freq"] = prefix_df["prefix"].map(prefix_frequencies)
prefix_df = prefix_df.drop_duplicates(subset = ["prefix"])
prefix_df = prefix_df[prefix_df["prefix"].isin(["N","S","E","W","NE","NW","SE","SW"])]
prefix_df = prefix_df.sort_values(by = "prefix_freq", ascending= False)
prefix_df
```

### Prefix/suffix collision

**Problem:** A street with a directional at both ends would be parsed wrong.  
**Check:** Rows with a directional prefix **and** suffix.  
**Result:** None → no handling needed

```python
## Check for collision where there are any prefixes or suffixes
suffix_and_prefix_df = csv.copy()
suffix_and_prefix_df["suffix"] = suffix_and_prefix_df["StreetName"].str.split().str[-1]
suffix_and_prefix_df["prefix"] = suffix_and_prefix_df["StreetName"].str.split().str[0]
suffix_and_prefix_df = suffix_and_prefix_df[
    suffix_and_prefix_df["suffix"].isin(["N","S","E","W","NE","NW","SE","SW"]) &
    suffix_and_prefix_df["prefix"].isin(["N","S","E","W","NE","NW","SE","SW"])
]

print(suffix_and_prefix_df)
#* Result shows that we don't need to worry about prefix/suffix collision
```

### Hard-negative score factors

**Problem:** It was unclear which bonus in the hard-negative score drives acceptance.  
**Check:** % of candidates passing 0.85 with each bonus removed.  
**Result:** All 0.397; without same-address 0.174; without street-type 0.320; directional / EXT have almost no effect

```python
# Checking on a per match_basis, what contributed and stacked up

# for aid, bid, jaro_score, address_score, dir_score, str_score, is_ext_score, total_score in debug_jw[:5]:
#     row1 = new_df[new_df["id"] == aid].iloc[0]
#     row2 = new_df[new_df["id"] == bid].iloc[0]
#     display(
#         new_df[(new_df["id"] == aid) | (new_df["id"] == bid)].head()
#         )
#     print("Summary")
#     print("jaro_score", jaro_score)
#     print("address_score", address_score)
#     print("dir_score", dir_score)
#     print("str_score", str_score)
#     print("is_ext_score", is_ext_score)
#     print("total_score", total_score)

    
#* Checking Each Factor on Address Acceptance Rate
num_pass_normal = sum(
    1 if total_score > 0.85 else 0 for aid, bid, jaro_score, address_score, dir_score, str_score, is_ext_score, total_score in debug_jw
)/len(debug_jw) * 100
print(f"{num_pass_normal=:.3f}")

num_pass_wo_address = sum(
    1 if total_score-address_score > 0.85 else 0 for aid, bid, jaro_score, address_score, dir_score, str_score, is_ext_score, total_score in debug_jw
)/len(debug_jw) * 100
print(f"{num_pass_wo_address=:.3f}")

num_pass_wo_dir = sum(
    1 if total_score-dir_score > 0.85 else 0 for aid, bid, jaro_score, address_score, dir_score, str_score, is_ext_score, total_score in debug_jw
)/len(debug_jw) * 100
print(f"{num_pass_wo_dir=:.3f}")

num_pass_wo_str = sum(
    1 if total_score-str_score > 0.85 else 0 for aid, bid, jaro_score, address_score, dir_score, str_score, is_ext_score, total_score in debug_jw
)/len(debug_jw) * 100
print(f"{num_pass_wo_str=:.3f}")

num_pass_wo_ext = sum(
    1 if total_score-is_ext_score > 0.85 else 0 for aid, bid, jaro_score, address_score, dir_score, str_score, is_ext_score, total_score in debug_jw
)/len(debug_jw) * 100
print(f"{num_pass_wo_ext=:.3f}")

#* Matplot graph, looking for any extreme outliers in the candidates
# import matplotlib.pyplot as plt
# lst = [[address, len(group_df)] for address, group_df in new_df.groupby("Address")]
# lst.sort(key = lambda x: [x[1], x[0]], reverse=True)
# print(lst[:5])

# selected_addresses_list = [new_df[new_df["id"] == a].iloc[0]["Address"] for a,b in all_scores_above(0.85)]
# values, counts = np.unique(selected_addresses_list, return_counts=True)
# freq_list = [[c, v] for v,c in zip(values, counts)]
# freq_list.sort(reverse = True)
# print(freq_list[:5])
# print(sum(c for c, v in freq_list[:10]) / 15435) #0.04742468415937804
# plt.hist(counts, bins = 100)
# plt.yscale("log")
# plt.xlabel("value")
# plt.title("")
# plt.show()

#* Sanity check looking at some of the matches
# for _ in range(4):
#     a, b = random.choice(train_df_lst)
#     row1 = new_df[new_df["id"] == a].iloc[0]
#     row2 = new_df[new_df["id"] == b].iloc[0]
#     tmp_df = pd.DataFrame([row1, row2])
#     display(tmp_df.head())
```
